<a href="https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmadhalawanii/flyrank-ml-capstone/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Freestyle — "Confound or Cause?"** *(continuing from ML-03 — same freestyle direction, now written as a real contract against the full warehouse instead of the 30k-row starter slice.)*

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis: one row = one pseudonymized content item (`content_hash_id`)** — described by `dim_content`'s time-invariant attributes (content type, authoring provider/model) and summarized by 30 days of `fact_content_daily_performance`.

**Time window: March 2026 (`report_date` in `month=2026-03`)** — one full mid-panel month, split at its midpoint into a **first half** (days 1–15, the only days I'm allowed to build features from) and a **second half** (days 16–31, where the label lives). I use March on purpose, not `fact_content_daily_performance_sample`: the sample table *is* June 2026, the panel's last month, so developing a decline label there would mean developing inside my own future test window.

Collapsing daily rows to one row per content item is deliberate: ML-03's question — is `model_used`'s association with decline real, or a stand-in for `content_type` / age — is about a time-invariant property of the item, not day-to-day noise. So the grain **I query from** is content-item-per-day; the grain **I model on** is content-item-per-month.

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# Confirm the tables/columns the claim above depends on -- don't assume names, check them.
for name in ['dim_content', 'fact_daily_mar']:
    print(f'--- {name} ---')
    print(con.sql(f"DESCRIBE SELECT * FROM {TABLES[name]}").df()[['column_name', 'column_type']].to_string(index=False))
    print()

MID = "DATE '2026-03-16'"  # splits first-half (feature window) from second-half (label window)

Paste your Hugging Face READ token (hf_...): ··········
--- dim_content ---
               column_name column_type
            client_hash_id     VARCHAR
           content_hash_id     VARCHAR
           keyword_hash_id     VARCHAR
               url_hash_id     VARCHAR
        keyword_char_count      BIGINT
       keyword_token_count      BIGINT
            url_char_count      BIGINT
      content_created_date        DATE
      content_updated_date        DATE
              content_type     VARCHAR
             search_volume      BIGINT
               competition      DOUBLE
         competition_level     VARCHAR
                       cpc      DOUBLE
               main_intent     VARCHAR
                 backlinks      BIGINT
            category_count      BIGINT
      keyword_created_date        DATE
             provider_used     VARCHAR
                model_used     VARCHAR
                char_count      BIGINT
                word_count      BIGINT
       last_optimized_date 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `content_type`, `provider_used`, `model_used` (`dim_content`) | Set when the content is authored — knowable months before March 2026. These are the confound and the candidate cause. |
| **Feature** | `content_age_days_at_window_start` (derived: days from `content_created_date` to `2026-03-01`) | Pure date arithmetic on a creation date already in the past. |
| **Feature** | `imp_first_half`, `pos_first_half` — first-half-March `gsc_impressions` / `gsc_avg_position` | A mid-month checkpoint anyone watching the account could see on March 16, before the second half happens. |
| **Label / proxy** | `is_declining` = second-half `gsc_impressions` total < first-half total | Defined FROM the second half by construction — can never also be a feature, same rule notebook 02 uses for `trend_pct`. |
| **Context** | `content_hash_id`, `client_hash_id`, `report_date` | Join / group / split keys only. `client_hash_id` also drives a client-grouped split so one client's pages can't sit in both halves of a comparison. |
| **Excluded** | `fact_content_query_90d` (whole table) | Out of scope for this question, and its 90-day window overlaps the very month I'm labeling — a leakage surface I don't need to take on for no added answer. |
| **Excluded** | Any row where the GA4/GSC availability flag is not `TRUE` | Three-valued flag (TRUE/FALSE/NULL, per the data dictionary) — FALSE or NULL means "not measured," not "zero engagement." |

In [2]:
fields = {
    'feature': ['content_type', 'provider_used', 'model_used',
                'content_age_days_at_window_start', 'imp_first_half', 'pos_first_half'],
    'label':   ['is_declining  (= imp_second_half < imp_first_half)'],
    'context': ['content_hash_id', 'client_hash_id', 'report_date'],
    'excluded': ["fact_content_query_90d (whole table) -- overlapping 90d window, out of scope for this question",
                 'rows where the GA4/GSC availability flag is not TRUE -- NULL/FALSE means not measured, not zero'],
}
for bucket, items in fields.items():
    print(f'{bucket.upper()}:')
    for i in items:
        print(f'  - {i}')
    print()

FEATURE:
  - content_type
  - provider_used
  - model_used
  - content_age_days_at_window_start
  - imp_first_half
  - pos_first_half

LABEL:
  - is_declining  (= imp_second_half < imp_first_half)

CONTEXT:
  - content_hash_id
  - client_hash_id
  - report_date

EXCLUDED:
  - fact_content_query_90d (whole table) -- overlapping 90d window, out of scope for this question
  - rows where the GA4/GSC availability flag is not TRUE -- NULL/FALSE means not measured, not zero



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query A — grain.** One row of `fact_content_daily_performance` should be one `(content_hash_id, report_date, client_hash_id)` combination — zero duplicates back means the grain holds.

**Query B — row count + span for my slice**, restricted to items with a real first-half signal (a 3-impression item swinging to 1 isn't a decline, it's noise).

**Query C — availability**, filtered with `IS TRUE` (never `= TRUE` / `= FALSE`), since the flag is three-valued. Checked for both `gsc_data_available` and `ga4_data_available` — the features below are GSC-based, so `gsc_data_available IS TRUE` is enforced directly in the feature-building query, not just measured here and left unapplied.

**Five features, one line each on why each is knowable at the decision moment (March 16):**
1. `content_type` — set at authoring time, months before March.
2. `provider_used` — an authoring choice, not a performance measurement.
3. `model_used` — the candidate cause itself; fixed before any March traffic exists.
4. `content_age_days_at_window_start` — date arithmetic between two already-past dates.
5. `imp_first_half` / `pos_first_half` — a mid-month checkpoint, observable on March 16 itself.

**The trap:** add the one column the label is computed from — `imp_second_half` — as a sixth feature on purpose, watch the score jump, then delete it.

In [3]:
# --- Query A: grain ---
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, client_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily_mar']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print(f'Query A -- duplicate-grain rows found: {len(grain_check)} (expect 0)')

# --- Query B: row count + span for my slice ---
MIN_FIRST_HALF_IMPRESSIONS = 100  # volume floor, same idea as notebook 01's impressions>=100 filter
slice_check = con.sql(f"""
    WITH bounds AS (SELECT {MID} AS mid),
    per_item AS (
        SELECT f.content_hash_id, SUM(f.gsc_impressions) AS imp_first_half
        FROM {TABLES['fact_daily_mar']} f, bounds b
        WHERE f.report_date < b.mid AND f.gsc_data_available IS TRUE
        GROUP BY 1
    )
    SELECT COUNT(*) AS n_items_over_floor, MIN(imp_first_half) AS min_imp, MAX(imp_first_half) AS max_imp
    FROM per_item WHERE imp_first_half >= {MIN_FIRST_HALF_IMPRESSIONS}
""").df()
print('Query B -- items above the volume floor:')
print(slice_check.to_string(index=False))

span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily_mar']}
""").df()
print('\nQuery B -- full slice span:')
print(span.to_string(index=False))

# --- Query C: availability, IS TRUE (three-valued flag) ---
# Both flags checked: features below use GSC columns, so gsc_data_available is the one that
# actually matters for this lane -- ga4_data_available is reported for completeness only.
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
           SUM(CASE WHEN gsc_data_available IS NULL THEN 1 ELSE 0 END) AS gsc_null_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
           SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_null_rows
    FROM {TABLES['fact_daily_mar']}
""").df()
print('\nQuery C -- availability:')
print(avail.to_string(index=False))
print(f"GSC: {avail['gsc_available_rows'][0] / avail['total_rows'][0]:.1%} survive IS TRUE; "
      f"{avail['gsc_null_rows'][0] / avail['total_rows'][0]:.1%} are NULL, not FALSE.")
print(f"GA4: {avail['ga4_available_rows'][0] / avail['total_rows'][0]:.1%} survive IS TRUE; "
      f"{avail['ga4_null_rows'][0] / avail['total_rows'][0]:.1%} are NULL, not FALSE.")

# --- Five-feature frame ---
features = con.sql(f"""
    WITH bounds AS (SELECT {MID} AS mid),
    dc AS (
        SELECT content_hash_id, content_type, provider_used, model_used, content_created_date
        FROM {TABLES['dim_content']}
    ),
    agg AS (
        SELECT f.content_hash_id, f.client_hash_id,
               SUM(CASE WHEN f.report_date < b.mid THEN f.gsc_impressions ELSE 0 END) AS imp_first_half,
               SUM(CASE WHEN f.report_date >= b.mid THEN f.gsc_impressions ELSE 0 END) AS imp_second_half,
               AVG(CASE WHEN f.report_date < b.mid THEN f.gsc_avg_position END) AS pos_first_half
        FROM {TABLES['fact_daily_mar']} f, bounds b
        WHERE f.gsc_data_available IS TRUE
        GROUP BY 1, 2
        HAVING imp_first_half >= {MIN_FIRST_HALF_IMPRESSIONS}
    )
    SELECT dc.content_hash_id, dc.content_type, dc.provider_used, dc.model_used,
           DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') AS content_age_days_at_window_start,
           agg.imp_first_half, agg.pos_first_half, agg.imp_second_half
    FROM agg JOIN dc USING (content_hash_id)
""").df()
features['is_declining'] = (features['imp_second_half'] < features['imp_first_half']).astype(int)

# provider_used has ~80% missing and its populated raw values include inconsistently formatted,
# apparent client-identifying strings -- bucket down to a safe, real category set BEFORE this
# ever hits get_dummies or gets printed anywhere that could reach a public commit.
KNOWN_PROVIDERS = {'google', 'openai'}
features['provider_used'] = features['provider_used'].where(
    features['provider_used'].isin(KNOWN_PROVIDERS) | features['provider_used'].isna(),
    'other'
)
print('provider_used after bucketing (safe to print -- no client strings):')
print(features['provider_used'].value_counts(dropna=False))

print(f'\n{len(features):,} content items in the feature frame')
print(features.head().to_string(index=False))

# --- The trap ---
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd

honest_cols = ['content_type', 'provider_used', 'model_used',
               'content_age_days_at_window_start', 'imp_first_half', 'pos_first_half']
X_honest = pd.get_dummies(features[honest_cols], columns=['content_type', 'provider_used', 'model_used'])
y = features['is_declining'].values

Xh_tr, Xh_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
tree_honest = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42).fit(Xh_tr, y_tr)
auc_honest = roc_auc_score(y_te, tree_honest.predict_proba(Xh_te)[:, 1])
print(f'\nHonest AUC (five features only): {auc_honest:.3f}')

X_leaky = X_honest.copy()
X_leaky['imp_second_half'] = features['imp_second_half'].values  # THE TRAP -- the exact number the label is computed from
Xl_tr, Xl_te, _, _ = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
tree_leaky = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42).fit(Xl_tr, y_tr)
auc_leaky = roc_auc_score(y_te, tree_leaky.predict_proba(Xl_te)[:, 1])
print(f"'Leaky' AUC (+ imp_second_half): {auc_leaky:.3f}  <- looks amazing, means nothing")
print(f'Jump from the leak: {auc_leaky - auc_honest:+.3f}')

del X_leaky  # delete the leaky column; keep only the honest number
print(f'\nKeeping the honest number: AUC = {auc_honest:.3f}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query A -- duplicate-grain rows found: 0 (expect 0)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query B -- items above the volume floor:
 n_items_over_floor  min_imp  max_imp
              77540    100.0 161575.0

Query B -- full slice span:
 n_rows  n_content_items   min_date   max_date
9841378           331437 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query C -- availability:
 total_rows  gsc_available_rows  gsc_null_rows  ga4_available_rows  ga4_null_rows
    9841378           3611061.0            0.0            413966.0      3018741.0
GSC: 36.7% survive IS TRUE; 0.0% are NULL, not FALSE.
GA4: 4.2% survive IS TRUE; 30.7% are NULL, not FALSE.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

provider_used after bucketing (safe to print -- no client strings):
provider_used
None      62043
google    13880
openai     1469
other       148
Name: count, dtype: int64

77,540 content items in the feature frame
         content_hash_id    content_type provider_used       model_used  content_age_days_at_window_start  imp_first_half  pos_first_half  imp_second_half  is_declining
content_04c67f3541177192 keyword article          None gemini-2.5-flash                               145           119.0       12.639599            212.0             0
content_1207efddce873942 keyword article          None gemini-2.5-flash                               145           283.0       12.587226            178.0             1
content_37952b007ab057b3 keyword article          None gemini-2.5-flash                               145           440.0       11.741716            334.0             1
content_7beb639d1052e49e keyword article          None gemini-2.5-flash                               145    

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced panel.** Per-client history depth varies widely — a single March-2026 slice over-represents clients onboarded early, and (verified below) not every client even has full reporting coverage within March itself.
- **Availability flags are three-valued, not boolean.** `= FALSE` instead of `IS TRUE` / `IS NOT TRUE` silently keeps NULL-flagged rows that look like zero engagement but are actually "not measured yet."
- **This stays observational, not causal**, even if `model_used` shows a real AUC lift. Content isn't randomly assigned to models — a lift says `model_used` carries independent signal, not that the model *caused* the difference.
- **One month is a short window for a 'decline' label.** A first-half vs second-half split inside March catches within-month movement, not a sustained multi-month trend — same caveat as notebook 01's Discovery C.
- **Named limitation for submission:** Client reporting coverage isn't uniform within my March window — of 55 clients, some report as few as 9 of 31 days, so for those clients `is_declining` may be comparing a real half-month against a mostly-empty one rather than two genuine two-week windows.

- **`provider_used` is unreliable.** ~80% missing (`None`), and the populated values include inconsistently formatted, client-identifying strings — bucketed to `google` / `openai` / `other` before use in section 3, and never printed unbucketed in this notebook.

In [4]:
# Supporting check for the 'unbalanced panel' limitation: even within one month,
# clients don't all have the same number of reporting days.
days_per_client = con.sql(f"""
    SELECT client_hash_id, COUNT(DISTINCT report_date) AS days_in_march
    FROM {TABLES['fact_daily_mar']}
    GROUP BY 1
""").df()
print(days_per_client['days_in_march'].describe())
print(f"\n{(days_per_client['days_in_march'] < 28).mean():.1%} of clients have fewer than 28 reporting days "
      "in March -- a client-level effect this contract does not correct for.")

count    55.000000
mean     29.890909
std       4.524308
min       9.000000
25%      31.000000
50%      31.000000
75%      31.000000
max      31.000000
Name: days_in_march, dtype: float64

5.5% of clients have fewer than 28 reporting days in March -- a client-level effect this contract does not correct for.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.